In [0]:
CATALOG = "energy_puls"
LANDING = f"/Volumes/{CATALOG}/landing/landing"
BRONZE = f"{CATALOG}.bronze"
CKPT = f"/Volumes/{CATALOG}/landing/_checkpoints"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")
print("Bronze schema ready:", BRONZE)

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing._checkpoints")
print("checkpoints volume ready")

In [0]:
from pyspark.sql import functions as F

(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("sep", ";")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{CKPT}/eco2mix_archive_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(f"{LANDING}/eco2mix/archive/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .option("checkpointLocation", f"{CKPT}/eco2mix_archive")
    .trigger(availableNow=True)
    .toTable(f"{BRONZE}.eco2mix_raw"))

print("eco2mix archive → Bronze done")

In [0]:
raw = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CKPT}/eco2mix_api_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(f"{LANDING}/eco2mix/api/"))

(raw.select(
        F.col("pull_timestamp"),
        F.explode("records").alias("r"),
        F.current_timestamp().alias("_ingested_at"),
        F.col("_metadata.file_path").alias("_source_file"))
    .select("pull_timestamp", "r.*", "_ingested_at", "_source_file")
    .writeStream
    .option("checkpointLocation", f"{CKPT}/eco2mix_api")
    .trigger(availableNow=True)
    .toTable(f"{BRONZE}.eco2mix_api_raw"))

print("eco2mix api → Bronze done")

In [0]:
display(spark.sql(f"""
  SELECT 'archive' AS src, COUNT(*) AS n,
         MIN(date_heure) AS mn, MAX(date_heure) AS mx
  FROM {BRONZE}.eco2mix_raw
  UNION ALL
  SELECT 'api', COUNT(*), MIN(date_heure), MAX(date_heure)
  FROM {BRONZE}.eco2mix_api_raw
"""))